In [1]:
using CSV
using DataFrames
using Images
using Languages
using MLDataUtils
using Plots
using TextAnalysis

In [2]:
DATA = "../../data"

"../../data"

In [3]:
df = DataFrame(CSV.File("$DATA/emails.csv"))

,Email No.,the,to,ect,and,for,of,a,you,hou,in
,String,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,Email 1,0,0,1,0,0,0,2,0,0,0
2,Email 2,8,13,24,6,6,2,102,1,27,18
3,Email 3,0,0,1,0,0,0,8,0,0,4
4,Email 4,0,5,22,0,5,1,51,2,10,1
5,Email 5,7,6,17,1,5,2,57,0,9,3
6,Email 6,4,5,1,4,2,3,45,1,0,16
7,Email 7,5,3,1,3,2,1,37,0,0,9
8,Email 8,0,2,2,3,1,2,21,6,0,2
9,Email 9,2,2,3,0,0,1,18,0,0,3


In [4]:
allwords = names(df)[2:end - 1]
allwords[1:10]

10-element Array{String,1}:
 "the"
 "to"
 "ect"
 "and"
 "for"
 "of"
 "a"
 "you"
 "hou"
 "in"

In [5]:
string([string("one", " "), string("two", " ")])

"[\"one \", \"two \"]"

In [6]:
allwordstext = StringDocument(
    string([string(word, " ") for word in allwords]...))

A StringDocument{String}
 * Language: Languages.English()
 * Title: Untitled Document
 * Author: Unknown Author
 * Timestamp: Unknown Time
 * Snippet: the to ect and for of a you hou in on is this enro

In [7]:
prepare!(allwordstext, strip_articles)
prepare!(allwordstext, strip_pronouns)
vocab = filter(x -> x != "", split(TextAnalysis.text(allwordstext)))
cleanwords_df = df[!, vocab]
datamatrix = Matrix(cleanwords_df)'

2975×5172 LinearAlgebra.Adjoint{Int64,Array{Int64,2}}:
 0  13  0   5   6   5   3   2   2   4  …   3  0  18  0  0   2  27   0   7  24
 1  24  1  22  17   1   1   2   3  35      1  1   3  1  1   2  11   1   1   5
 0   6  0   0   1   4   3   3   0   0      2  0   1  0  1   3   2   1   0   1
 0   6  0   5   5   2   2   1   0   1      1  0   6  3  0   0   6   0   2   6
 0   2  0   1   2   3   1   2   1   0      2  0   4  1  0   0   5   0   1   5
 0  27  0  10   9   0   0   0   0  16  …   0  0   2  0  0   0   3   0   0   2
 0  18  4   1   3  16   9   2   3   9      7  0  18  2  0   5  23   1   8  23
 0  21  2   5  12  12   4   6   3   4      3  0  11  2  0   6  18   1  11  13
 1  13  0   9   2   8   6   2   2   1      2  0   4  1  1   1   6   3   7   5
 0   0  0   2   2   1   2   0   1   0      0  0   3  0  0   0   3   1   1   4
 0   1  0   0   0   0   0   0   0   0  …   0  0   1  0  0   0   1   0   0   1
 2  61  8  16  30  52  27  28  15  35     26  1  70  7  1  20  98  10  39  99
 0   4  0

In [8]:
labels = df.Prediction

5172-element Array{Int64,1}:
 0
 0
 0
 0
 0
 1
 0
 1
 0
 0
 0
 0
 0
 ⋮
 0
 0
 1
 1
 0
 0
 1
 0
 0
 1
 1
 0

In [9]:
(Xtrain, ytrain), (Xtest, ytest) = splitobs(
    shuffleobs((datamatrix, labels)), at=0.7)

(([28 0 … 1 9; 13 1 … 1 7; … ; 0 0 … 0 0; 0 0 … 0 0], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0  …  0, 1, 0, 0, 0, 0, 1, 1, 0, 0]), ([0 3 … 4 2; 1 1 … 1 1; … ; 0 0 … 0 0; 0 0 … 0 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 1  …  1, 0, 0, 0, 0, 1, 0, 0, 1, 1]))

In [10]:
mutable struct BayesSpamFilter
    wordcountham::Dict{String, Int64}
    wordcountspam::Dict{String, Int64}
    nham::Int64
    nspam::Int64
    vocab::Array{String}
    BayesSpamFilter() = new()
end

In [11]:
function countwords(worddata, vocab, labels, spam=0)
    # worddata is a matrix where each col is an email and each row a word
    countdict = Dict{String, Int64}()
    nemails = size(worddata)[2]
    for (i, word) in enumerate(vocab)
        countdict[word] = sum(
            [worddata[i, j] for j in 1:nemails if labels[j] == spam])
    end
    countdict
end

countwords (generic function with 2 methods)

In [12]:
function fit!(model::BayesSpamFilter, Xtrain, ytrain, vocab)
    model.vocab = vocab
    model.wordcountham = countwords(Xtrain, model.vocab, ytrain, 0)
    model.wordcountspam = countwords(Xtrain, model.vocab, ytrain, 1)
    model.nham = sum(values(model.wordcountham))
    model.nspam = sum(values(model.wordcountspam))
    return
end

fit! (generic function with 1 method)

In [13]:
spamfilter = BayesSpamFilter()
fit!(spamfilter, Xtrain, ytrain, vocab)

In [14]:
function get_hamspamprob(
        word, wordcountham, wordcountspam, nham, nspam, nvocab, alpha)
    hamprob = (wordcountham[word] + alpha) / (nham + alpha*nvocab)
    spamprob = (wordcountspam[word] + alpha) / (nspam + alpha*nvocab)
    hamprob, spamprob
end

get_hamspamprob (generic function with 1 method)

In [15]:
function predictspam(email, model::BayesSpamFilter, alpha, tol=100)
    ngrams_email = ngrams(StringDocument(email))
    emailwords = keys(ngrams_email)
    nvocab = length(model.vocab)
    mod_nemail = model.nham + model.nspam
    hamprior = model.nham / mod_nemail
    spamprior = model.nspam / mod_nemail
    if length(emailwords) > tol
        wordfreq = values(ngrams_email)
        sortidx = sortperm(collect(wordfreq), rev=true)
        emailwords = collect(emailwords)[sortidx][1:tol]
    end
    email_hamprob = BigFloat(1)
    email_spamprob = BigFloat(1)
    for word in [w for w in emailwords if w in vocab]
        wordhamprob, wordspamprob = get_hamspamprob(
            word, 
            model.wordcountham, 
            model.wordcountspam, 
            model.nham, 
            model.nspam, 
            nvocab, 
            alpha)
        email_hamprob *= wordhamprob
        email_spamprob *= wordspamprob
    end
    hamprior * email_hamprob, spamprior * email_spamprob
end

predictspam (generic function with 2 methods)

In [16]:
# Test
TEST_EMAIL = 56
email = string(
    [repeat(string(word, " "), n) 
     for (word, n) in zip(vocab, Xtest[:, TEST_EMAIL])]...)
hamprob, spamprob = predictspam(email, spamfilter, 1)

(7.612196036286371686157307601478590097076861352724625343689551930290735948990174e-266, 3.275192062948620881986327872137621264910945583564581195438458389853483548341178e-297)

In [17]:
argmax([hamprob, spamprob]) # ham

1

In [18]:
ytest[TEST_EMAIL] # 1 = spam

0

In [27]:
function predictall(Xtest, ytest, model::BayesSpamFilter, alpha, tol=200)
    n = length(ytest)
    preds = Array{Int64, 1}(undef, n)
    for i in 1:n
        email = string(
            [repeat(string(word, " "), n)
             for (word, n) in zip(model.vocab, Xtest[:, i])]...)
        pham, pspam = predictspam(email, model, alpha, tol)
        pred = argmax([pham, pspam]) - 1
        preds[i] = pred
    end
    preds
end

predictall (generic function with 3 methods)

In [28]:
preds = predictall(Xtest, ytest, spamfilter, 1)

1552-element Array{Int64,1}:
 0
 0
 0
 0
 1
 0
 0
 0
 0
 1
 0
 1
 1
 ⋮
 1
 0
 1
 0
 0
 0
 0
 1
 0
 0
 1
 1

In [32]:
function get_accuracy(preds, actual)
    n = length(preds)
    correct = sum(preds .== actual)
    accuracy = correct / n
    accuracy
end

get_accuracy (generic function with 3 methods)

In [33]:
get_accuracy(preds, ytest)

0.9458762886597938